## Serving Open LLMs with Ollama

[**Ollama**](https://ollama.com) is a tool for running and managing large language models. Ollamaprovides...
- an engine for efficiently storing, loading, running (on CPU and GPU), and unloading large language models
- a command line interface for model management: creating, running, pulling, and deleting models.
- a REST API for programmatically running and managing models, ideal for integrating into existing applications or services.
- a [library of models](https://ollama.com/library)
- an option to import custom models using a `Modelfile`


In [ ]:
from ai_dojo import show

In [ ]:
show.github_repo("https://github.com/ollama/ollama")

## Setup

Follow the installation steps lined out in the ollama repo to install it and start it on your system. Alternatively, if you have docker, you can just do

```
docker run -d -v ollama:/root/.ollama -p 11434:11434 --name ollama ollama/ollama
``` 

## Python Library 

Instead of using the CLI or the REST API directly, we can use this Python module to interact with the Ollama API. It basically translates the API to Python function calls.

In [ ]:
show.github_repo("https://github.com/ollama/ollama-python")

In [ ]:
import ollama

## Pulling Models

We download and store a model via the `pull` command.

Take for example a small chat model that we found in the Ollama library:
https://ollama.com/library/tinyllama:chat

In [ ]:
ollama_model_name = "tinyllama:chat"

In [ ]:
ollama.pull(ollama_model_name)

After we have pulled the model, it is available locally. Let's use the `chat` interface to prompt it.

In [ ]:
response = ollama.chat(
    model=ollama_model_name,
    messages=[
        {
            "role": "user",
            "content": "What is the meaning of life? Give a very short answer!", 
        },
    ],
    stream=True,
)
show.stream(response)

How are such models stored on disk? A common format is GGUF.

<div style="border:2px solid gray; padding:10px; width: 95%;">

**ℹ The GGUF File Format**

[**GGUF**](https://huggingface.co/docs/hub/gguf) is a binary format that is optimized for quick loading and saving of models, making it highly efficient for inference purposes. Unlike tensor-only file formats like safetensors, GGUF encodes both the tensors and a standardized set of metadata. _(Adapted from the [Hugging Face Hub documentation](https://huggingface.co/docs/hub/gguf).)_


<div style="text-align: center; margin: 30px 0;">
  <img src="graphics/gguf-spec.png" style="max-width: 800px; max-height: 600px; border: 2px solid #ddd; border-radius: 8px; padding: 5px; box-shadow: 0 2px 8px rgba(0,0,0,0.1);">
  <div style="margin-top: 10px; font-style: italic; color: #666; font-size: 0.9em;">
    GGUF file format — diagram by <a href="https://github.com/mishig25">@mishig25</a>, <a href="https://huggingface.co/datasets/huggingface/documentation-images">Hugging Face documentation images</a>, <a href="https://creativecommons.org/licenses/by-nc-sa/4.0/">CC BY-NC-SA 4.0</a>
  </div>
</div>

## Model Overview

The `list` command is there to list all locally available models. 

In [ ]:
ollama.list()

... but let's bring the output into a nicer form for better overview:

In [ ]:
import pandas
from pandas import DataFrame

In [ ]:
def show_ollama_model_table() -> DataFrame:
    """List Ollama models in tabular form"""
    response = ollama.list()
    # The new API returns a ListResponse object with .models attribute
    models = response.models
    # Convert Model objects to dictionaries for pandas
    model_dicts = []
    for m in models:
        model_dicts.append({
            "model": m.model,
            "size (GB)": round(m.size / (1024**3), 2),
            "family": m.details.family if m.details else None,
            "format": m.details.format if m.details else None,
            "parameter_size": m.details.parameter_size if m.details else None,
            "quantization_level": m.details.quantization_level if m.details else None,
        })
    return pandas.DataFrame(model_dicts)

In [ ]:
show_ollama_model_table()

## Calling a Remote Ollama Instance

Ollama can easily function as a remote LLM server. Simply create a client and configure it with the host name of the remote machine and the port that Ollama is listening on.

In [ ]:
from ollama import Client
from ai_dojo.lm import get_ollama_url

client = Client(
    host=get_ollama_url(),  # configured via OLLAMA_HOST env var or set custom URL
    timeout=30,             # seconds to wait for client to be reachable
)


In [ ]:
client.list()

In [ ]:
response = client.chat(
    model="qwen3.5:1.7b",
    messages=[
        {
            "role": "user",
            "content": "What is the meaning of life? Give a very short answer!", 
        },
    ],
    stream=True,
)
show.stream(response)

---
_This notebook is licensed under a [Creative Commons Attribution-NonCommercial-ShareAlike 4.0 International (CC BY-NC-SA 4.0)](https://creativecommons.org/licenses/by-nc-sa/4.0/). Copyright © 2024-2026 [Christian Staudt](https://clstaudt.me), [Katharina Rasch](https://krasch.io)_